# 투구 제구 성공 확률 모델 V2.1 학습

V1 Legacy Global을 복원하고 R/F Expert와 Pitcher Expert를 결합합니다. 최근 연도 가중 R/F별 Brier 블렌딩과 walk-forward 보정을 적용하며 테스트 행끼리 집계하지 않습니다.

In [1]:
import importlib.util
import importlib.metadata
import subprocess
import sys

required_packages = {
    "catboost": ("catboost", "1.2.10"),
    "sklearn": ("scikit-learn", "1.8.0"),
    "pyarrow": ("pyarrow", "25.0.1"),
}
install = []
for module, (distribution, wanted) in required_packages.items():
    found = importlib.util.find_spec(module) is not None
    current = importlib.metadata.version(distribution) if found else None
    if current != wanted:
        install.append(f"{distribution}=={wanted}")
if install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *install])


In [2]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display
from train_v21 import run_training

ROOT = Path.cwd().resolve()
if not (ROOT / "result.py").exists():
    raise FileNotFoundError("프로젝트 루트에서 main.ipynb를 실행해 주세요.")
RUN_DIR = Path(os.environ.get("BASEBALL_RUN_DIR", ROOT)).resolve()
print(f"project={ROOT}, run_dir={RUN_DIR}")
print(f"task_type={os.environ.get('BASEBALL_TASK_TYPE', 'GPU')}, devices={os.environ.get('BASEBALL_GPU_DEVICES', '0')}")


project=D:\baseball, run_dir=D:\baseball
task_type=GPU, devices=0


## V2.1 학습 및 OOF 평가

기본값은 전체 데이터와 GPU 0번입니다. 빠른 점검만 할 때는 실행 전에 `BASEBALL_FAST_MODE=1`을 설정하세요. FAST_MODE 결과는 제출에 사용하면 안 됩니다.

In [3]:
result = run_training(ROOT, RUN_DIR)
display(result["metrics"].sort_values(["season", "model"]))
print("Selected R expert:", result["summary"]["selected_regular"])
print("Selected F expert:", result["summary"]["selected_futures"])
print("Final weights by type:", result["summary"]["ensemble"]["weight_map_by_game_type"])
print("Calibration:", result["summary"]["calibration"])


Default metric period is 5 because BrierScore is/are not implemented for GPU
Metric BrierScore is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.2495552	test: 0.2494742	best: 0.2494742 (0)	total: 121ms	remaining: 2m 25s
100:	learn: 0.2428164	test: 0.2434661	best: 0.2434658 (98)	total: 11.4s	remaining: 2m 3s
200:	learn: 0.2418054	test: 0.2434999	best: 0.2434590 (109)	total: 22.5s	remaining: 1m 51s
bestTest = 0.2434590455
bestIteration = 109
Shrink model to first 110 iterations.
0:	learn: 0.6923494	test: 0.6926744	best: 0.6926744 (0)	total: 54.1ms	remaining: 1m 4s
100:	learn: 0.6806843	test: 0.6815133	best: 0.6815076 (95)	total: 9.41s	remaining: 1m 42s
200:	learn: 0.6786348	test: 0.6816441	best: 0.6814702 (130)	total: 15.6s	remaining: 1m 17s
bestTest = 0.6814702143
bestIteration = 130
Shrink model to first 131 iterations.
0:	learn: 0.6927225	test: 0.6929383	best: 0.6929383 (0)	total: 93.7ms	remaining: 1m 52s
100:	learn: 0.6842275	test: 0.6903741	best: 0.6903741 (100)	total: 10.1s	remaining: 1m 49s
200:	learn: 0.6820413	test: 0.6904891	best: 0.6903694 (105)	total: 20s	remaining: 1m 39s
bestTest = 0.6903694263
bestItera

Default metric period is 5 because BrierScore is/are not implemented for GPU
Metric BrierScore is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.2495285	test: 0.2499824	best: 0.2499824 (0)	total: 251ms	remaining: 5m
100:	learn: 0.2425751	test: 0.2530934	best: 0.2499770 (1)	total: 14.6s	remaining: 2m 38s
bestTest = 0.2499769829
bestIteration = 1
Shrink model to first 2 iterations.
0:	learn: 0.6922473	test: 0.6930551	best: 0.6930551 (0)	total: 62ms	remaining: 1m 14s
100:	learn: 0.6797136	test: 0.6961381	best: 0.6929383 (7)	total: 6.79s	remaining: 1m 13s
bestTest = 0.6929382828
bestIteration = 7
Shrink model to first 8 iterations.
0:	learn: 0.6928024	test: 0.6929855	best: 0.6929855 (0)	total: 213ms	remaining: 4m 15s
100:	learn: 0.6852641	test: 0.6912699	best: 0.6911746 (72)	total: 13.2s	remaining: 2m 24s
bestTest = 0.6911746352
bestIteration = 72
Shrink model to first 73 iterations.
0:	learn: 0.6927724	test: 0.6929586	best: 0.6929586 (0)	total: 202ms	remaining: 4m 1s
100:	learn: 0.6850569	test: 0.6912289	best: 0.6912005 (84)	total: 13.1s	remaining: 2m 22s
200:	learn: 0.6831673	test: 0.6915464	best: 0.6912005 (84)	total

Default metric period is 5 because BrierScore is/are not implemented for GPU
Metric BrierScore is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.2496379	test: 0.2499039	best: 0.2499039 (0)	total: 339ms	remaining: 6m 47s
100:	learn: 0.2440227	test: 0.2481050	best: 0.2481050 (100)	total: 19.1s	remaining: 3m 27s
200:	learn: 0.2431419	test: 0.2481324	best: 0.2480863 (111)	total: 37.9s	remaining: 3m 8s
bestTest = 0.2480863149
bestIteration = 111
Shrink model to first 112 iterations.
0:	learn: 0.6925375	test: 0.6930162	best: 0.6930162 (0)	total: 77.4ms	remaining: 1m 32s
100:	learn: 0.6832476	test: 0.6932380	best: 0.6925151 (21)	total: 8.66s	remaining: 1m 34s
bestTest = 0.6925151475
bestIteration = 21
Shrink model to first 22 iterations.
0:	learn: 0.6928458	test: 0.6929767	best: 0.6929767 (0)	total: 311ms	remaining: 6m 13s
100:	learn: 0.6858378	test: 0.6903225	best: 0.6903146 (94)	total: 16.9s	remaining: 3m 3s
200:	learn: 0.6836126	test: 0.6904024	best: 0.6903090 (127)	total: 33s	remaining: 2m 43s
bestTest = 0.6903090422
bestIteration = 127
Shrink model to first 128 iterations.
0:	learn: 0.6928437	test: 0.6929271	best: 0.6

Default metric period is 5 because BrierScore is/are not implemented for GPU
Metric BrierScore is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.2497243	total: 126ms	remaining: 7.06s
56:	learn: 0.2457907	total: 2.78s	remaining: 0us
0:	learn: 0.6926985	total: 40.9ms	remaining: 573ms
14:	learn: 0.6889775	total: 626ms	remaining: 0us
0:	learn: 0.6928874	total: 107ms	remaining: 10.6s
99:	learn: 0.6872146	total: 4.46s	remaining: 0us
0:	learn: 0.6924137	total: 18.4ms	remaining: 1.27s
69:	learn: 0.6718569	total: 1.25s	remaining: 0us
Saved D:\baseball\output\submission.csv | rows=5 mean=0.463359 min=0.424630 max=0.493609
V2.1 training complete in 711.9s


,season,model,rows,brier,brier_skill_train_prior,log_loss,roc_auc,target_mean,prediction_mean,best_iteration
5,2022,futures_hl1,30448,0.210606,0.099393,0.612351,0.520029,0.708749,0.650031,420
6,2022,futures_post2023,30448,0.210606,0.099393,0.612351,0.520029,0.708749,0.650031,420
7,2022,futures_weak_old,30448,0.211741,0.094540,0.614880,0.526230,0.708749,0.635945,242
25,2022,game_type,247472,0.243938,0.021767,0.680771,0.576955,0.528920,0.526833,0
24,2022,legacy_global,247472,0.243459,0.023687,0.679724,0.577999,0.528920,0.534456,0
26,2022,pitcher,247472,0.244268,0.020443,0.681470,0.574838,0.528920,0.528462,0
2,2022,regular_hl1_5,217024,0.248614,0.011642,0.690370,0.542434,0.503691,0.509548,106
3,2022,regular_hl2_5,217024,0.248582,0.011770,0.690305,0.543201,0.503691,0.509828,89
4,2022,regular_uniform,217024,0.248677,0.011393,0.690495,0.541294,0.503691,0.509641,136
27,2022,walk_forward_blend,247472,0.243578,0.023210,0.680003,0.578292,0.528920,0.530970,0


Selected R expert: regular_hl1_5
Selected F expert: futures_post2023
Final weights by type: {'R': {'legacy_global': 0.5264441941057806, 'game_type': 0.47355580589421953, 'pitcher': 0.0}, 'F': {'legacy_global': 0.7025530443716325, 'game_type': 0.29744695562836765, 'pitcher': 0.0}}
Calibration: {'version': 3, 'method': 'affine', 'selection_scores': {'identity': 0.2477118268609047, 'affine': 0.24764372408390045, 'type_affine': 0.24782317578792573}, 'trained_on_seasons': [2022, 2023, 2024], 'slope': 1.18455002043331, 'intercept': -0.09989944683928176}


In [4]:
display(result["correlations"])
display(result["importance"].head(25))
display(pd.read_csv(result["submission_path"]))
print("다음으로 evaluation.ipynb를 실행해 세부 평가를 확인하세요.")


,legacy_global,game_type,pitcher
legacy_global,1.000000,0.857376,0.882493
game_type,0.857376,1.000000,0.801445
pitcher,0.882493,0.801445,1.000000


,feature,legacy_global,pitcher_expert,regular_expert,futures_expert,mean_importance
84,recent_game_type_prior,0.000000,47.005224,0.046716,0.000000,11.762985
98,season,24.810511,0.000000,3.912291,0.000000,7.180700
35,game_type,22.069901,0.000000,0.000000,0.000000,5.517475
14,asof_pitcher_prev5_game_success_rate,3.723456,5.268488,5.396597,4.580135,4.742169
17,asof_pitcher_success_rate,7.405427,5.831462,3.376693,2.034391,4.661993
3,asof_pitcher_ball_rate,2.971551,7.659332,5.256555,2.056177,4.485903
95,same_hand,4.081681,0.000000,5.356442,7.220696,4.164705
62,pitcher_id,1.968146,0.000000,8.659346,2.863092,3.372646
15,asof_pitcher_reverse_rate,2.942965,2.407331,4.644167,2.641890,3.159088
36,hand_matchup_code,2.774951,0.000000,3.192715,5.891728,2.964848


,row_id,control_success
0,TEST_000001,0.439244
1,TEST_000017,0.424630
2,TEST_000213,0.474026
3,TEST_005332,0.493609
4,TEST_035185,0.485285


다음으로 evaluation.ipynb를 실행해 세부 평가를 확인하세요.
